# **Day 8.2: Hyperparameter Tuning - Optimizing Model Performance**

## **Table of Contents**
1. [Learning Objectives](#learning-objectives)
2. [8.2.1 What are Hyperparameters?](#821-what-are-hyperparameters)
3. [8.2.2 Manual Search vs. Systematic Search](#822-manual-search-vs-systematic-search)
4. [8.2.3 Grid Search (GridSearchCV)](#823-grid-search-gridsearchcv)
5. [8.2.4 Randomized Search (RandomizedSearchCV)](#824-randomized-search-randomizedsearchcv)
6. [8.2.5 Best Practices & Computational Considerations](#825-best-practices--computational-considerations)
7. [Summary & Transition to Note 8.3](#summary--transition-to-note-83)

## **Learning Objectives**
By the end of this section, you will be able to:
- Distinguish between model parameters and hyperparameters
- Understand why hyperparameter tuning is crucial for optimal performance
- Implement Grid Search for systematic hyperparameter optimization
- Use Randomized Search for efficient exploration of large parameter spaces
- Choose appropriate search strategies based on computational constraints
- Avoid common pitfalls in hyperparameter tuning

## **8.2.1 What are Hyperparameters?**

Building on your solid foundation from Days 1-7, let's distinguish between **parameters** and **hyperparameters**.

### **Parameters vs. Hyperparameters**

**Parameters** (learned from data):
- **Linear Regression:** Coefficients (slopes, intercepts) learned during fitting
- **Decision Trees:** The actual split conditions learned from training data
- **Logistic Regression:** Weights learned to separate classes
- **Random Forest:** Individual tree structures built from training data

**Hyperparameters** (set before training):
- **Decision Trees:** `max_depth`, `min_samples_split`, `min_samples_leaf`
- **Random Forest:** `n_estimators`, `max_depth`, `max_features`
- **Logistic Regression:** `C` (regularization strength), `solver`, `penalty`
- **K-Nearest Neighbors:** `n_neighbors`, `weights`, `metric`

### **Key Characteristics of Hyperparameters**

1. **Set Before Training:** You must choose them before calling `.fit()`
2. **Control Learning Process:** They determine how the algorithm learns
3. **Control Model Complexity:** They often regulate overfitting vs. underfitting
4. **Algorithm-Specific:** Different algorithms have different hyperparameters

### **Real-World Example: Random Forest**

In [1]:
from sklearn.ensemble import RandomForestClassifier

# These are all hyperparameters you set:
rf = RandomForestClassifier(
    n_estimators=100,        # How many trees to build
    max_depth=10,           # Maximum depth of each tree
    min_samples_split=5,    # Min samples required to split a node
    min_samples_leaf=2,     # Min samples required at leaf nodes
    max_features='sqrt',    # Features to consider for each split
    random_state=42         # For reproducibility
)

# During .fit(), the algorithm learns:
# - Which features to split on at each node
# - What split values to use
# - The structure of each individual tree

### **Why Do Hyperparameters Matter?**

**Default values are rarely optimal** for your specific dataset. Consider this example:

In [ ]:
# Default Random Forest
rf_default = RandomForestClassifier(random_state=42)
# Might give you 85% accuracy

# Tuned Random Forest  
rf_tuned = RandomForestClassifier(
    n_estimators=200, 
    max_depth=15, 
    min_samples_split=10,
    random_state=42
)
# Might give you 92% accuracy!

That's a **7% improvement** just from better hyperparameter choices!

### **Knowledge Check Questions (8.2.1)**

1. **Classification:** For each example, identify whether it's a parameter or hyperparameter:
   - The coefficient for "age" in a linear regression model
   - The `n_neighbors=5` setting in KNN
   - The specific split condition "income > $50k" learned by a decision tree
   - The `C=1.0` regularization strength in logistic regression

2. **Practical Understanding:** You're using a Random Forest and getting 78% accuracy with default settings. Your colleague suggests "just increase `n_estimators`." Why might this help, and what are the potential downsides?

3. **Algorithm Connection:** Think back to Day 6 (Decision Trees). How do the hyperparameters `max_depth` and `min_samples_split` help prevent overfitting?

## **8.2.2 Manual Search vs. Systematic Search**

Now that you understand what hyperparameters are, how do you find the best values?

### **Manual Search (Trial and Error)**

This is what many beginners do:

In [ ]:
# Try 1
rf1 = RandomForestClassifier(n_estimators=50, max_depth=5)
# CV Score: 0.82

# Try 2  
rf2 = RandomForestClassifier(n_estimators=100, max_depth=10)
# CV Score: 0.85

# Try 3
rf3 = RandomForestClassifier(n_estimators=200, max_depth=15)
# CV Score: 0.87

# Keep going manually...

**Problems with Manual Search:**
- **Time-consuming:** You're guessing blindly
- **Inconsistent:** No systematic exploration
- **Suboptimal:** You might miss the best combinations
- **Biased:** You tend to try "nice" numbers (50, 100, 200)

### **Systematic Search Strategies**

**Better approaches** that use the cross-validation you learned in Note 8.1:

1. **Grid Search:** Try all combinations in a predefined grid
2. **Randomized Search:** Sample combinations randomly
3. **Advanced Methods:** Bayesian optimization, genetic algorithms (beyond this course)

### **The Role of Cross-Validation**

**Critical point:** Hyperparameter tuning must use cross-validation to avoid overfitting to your test set!

In [ ]:
# ❌ WRONG: This will overfit to test set
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Try different hyperparameters
rf1.fit(X_train, y_train); score1 = rf1.score(X_test, y_test)  # 0.85
rf2.fit(X_train, y_train); score2 = rf2.score(X_test, y_test)  # 0.87
rf3.fit(X_train, y_train); score3 = rf3.score(X_test, y_test)  # 0.89

# Choose rf3 because it scored best on test set
# But now your "test" set isn't truly unseen anymore!

In [ ]:
# ✅ CORRECT: Use CV for hyperparameter selection
from sklearn.model_selection import cross_val_score

# Use only training data for hyperparameter tuning
cv_score1 = cross_val_score(rf1, X_train, y_train, cv=5).mean()  # 0.83
cv_score2 = cross_val_score(rf2, X_train, y_train, cv=5).mean()  # 0.85  
cv_score3 = cross_val_score(rf3, X_train, y_train, cv=5).mean()  # 0.86

# Choose rf3 based on CV scores
# Test set remains unbiased for final evaluation

### **Knowledge Check Questions (8.2.2)**

1. **Problem Identification:** Why is manual hyperparameter tuning often inefficient and potentially biased?

2. **CV Application:** Explain why cross-validation is essential for hyperparameter tuning. What happens if you use the test set instead?

3. **Strategy Comparison:** You have 3 hyperparameters, each with 5 possible values. How many combinations would you need to try with grid search? Why might randomized search be attractive for larger search spaces?

## **8.2.3 Grid Search (GridSearchCV)**

**Grid Search** is a systematic method that tries every possible combination of hyperparameters you specify.

### **How Grid Search Works**

1. **Define Parameter Grid:** Specify which hyperparameters to tune and their possible values
2. **Generate All Combinations:** Create every possible combination 
3. **Cross-Validate Each:** Use CV to evaluate each combination
4. **Select Best:** Choose the combination with the highest CV score
5. **Retrain:** Train final model on full training set with best parameters

### **Visual Example: 2D Grid**
```
     max_depth=5    max_depth=10    max_depth=15
n_est=50    [CV: 0.82]   [CV: 0.84]    [CV: 0.83]
n_est=100   [CV: 0.85]   [CV: 0.87]    [CV: 0.86]  ← Best!
n_est=200   [CV: 0.84]   [CV: 0.86]    [CV: 0.85]
```

### **Scikit-learn Implementation**

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

# Load and split data
data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, 
                                                    random_state=42, stratify=y)

# Step 1: Define the model
rf = RandomForestClassifier(random_state=42)

# Step 2: Define parameter grid
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, 15, None],
    'min_samples_split': [2, 5, 10]
}
# This creates 3 × 4 × 3 = 36 combinations to try

# Step 3: Create GridSearchCV object
grid_search = GridSearchCV(
    estimator=rf,                # The model to tune
    param_grid=param_grid,       # Parameter combinations to try
    cv=5,                        # 5-fold cross-validation
    scoring='accuracy',          # Metric to optimize
    n_jobs=-1,                   # Use all CPU cores
    verbose=1                    # Show progress
)

# Step 4: Fit GridSearchCV (this does all the work!)
print("Starting Grid Search...")
grid_search.fit(X_train, y_train)
print("Grid Search completed!")

# Step 5: Get results
print("\nBest Parameters:")
print(grid_search.best_params_)

print(f"\nBest Cross-Validation Score: {grid_search.best_score_:.4f}")

print("\nTop 5 combinations:")
results_df = pd.DataFrame(grid_search.cv_results_)
top_5 = results_df.nlargest(5, 'mean_test_score')[['params', 'mean_test_score', 'std_test_score']]
print(top_5)

### **Evaluating the Best Model**

In [ ]:
# The best model is automatically retrained on full training set
best_model = grid_search.best_estimator_

# Evaluate on held-out test set
test_score = best_model.score(X_test, y_test)
print(f"\nFinal Test Set Performance: {test_score:.4f}")

# Compare with default model
default_rf = RandomForestClassifier(random_state=42)
default_rf.fit(X_train, y_train)
default_score = default_rf.score(X_test, y_test)

print(f"Default RF Test Score: {default_score:.4f}")
print(f"Tuned RF Test Score: {test_score:.4f}")
print(f"Improvement: {test_score - default_score:.4f}")

### **Analyzing Results**

In [ ]:
# Understanding the search results
import pandas as pd

results = pd.DataFrame(grid_search.cv_results_)

# See all combinations and their scores
print("All parameter combinations tested:")
for i in range(len(results)):
    params = results.loc[i, 'params']
    score = results.loc[i, 'mean_test_score']
    std = results.loc[i, 'std_test_score']
    print(f"{params} → CV: {score:.4f} ± {std:.4f}")

### **Knowledge Check Questions (8.2.3)**

1. **Calculation Practice:** You define a grid with 4 values for `n_estimators`, 3 values for `max_depth`, and 2 values for `min_samples_split`. If you use 5-fold CV, how many total model fits will GridSearchCV perform?

2. **Result Interpretation:** Your GridSearchCV finds best parameters: `{'n_estimators': 200, 'max_depth': 10}` with CV score 0.87. The default model gets 0.82 on the test set, and your tuned model gets 0.84 on the test set. Is this concerning? Why might the test score be lower than the CV score?

3. **Parameter Selection:** You're tuning a Decision Tree and considering these ranges:
   - `max_depth`: [1, 5, 10, 20, None]
   - `min_samples_split`: [2, 10, 50, 100]
   
   What potential issues do you see with these ranges? How would you improve them?


## **8.2.4 Randomized Search (RandomizedSearchCV)**

While Grid Search is systematic, it can be computationally expensive. **Randomized Search** offers a more efficient alternative.

### **The Problem with Grid Search**

As your search space grows, Grid Search becomes impractical:

In [ ]:
# Large grid example
param_grid = {
    'n_estimators': [50, 100, 150, 200, 250, 300],      # 6 values
    'max_depth': [3, 5, 7, 10, 15, 20, None],           # 7 values  
    'min_samples_split': [2, 5, 10, 15, 20],            # 5 values
    'min_samples_leaf': [1, 2, 4, 6, 8],                # 5 values
    'max_features': ['sqrt', 'log2', None]              # 3 values
}
# Total combinations: 6 × 7 × 5 × 5 × 3 = 3,150 combinations!
# With 5-fold CV: 15,750 model fits!

### **How Randomized Search Works**

Instead of trying all combinations, Randomized Search:
1. **Samples randomly** from the parameter space
2. **Tries a fixed number** of combinations (you choose how many)
3. **Uses the same CV evaluation** as Grid Search
4. **Often finds good solutions** in much less time

### **Key Insight: Most Parameters Don't Matter Much**

Research shows that typically only a few hyperparameters significantly impact performance. Randomized Search is effective because it explores the important dimensions while ignoring less relevant ones.

### **Scikit-learn Implementation**

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

# Method 1: Using lists (similar to GridSearch)
param_dist = {
    'n_estimators': [50, 100, 150, 200, 250, 300],
    'max_depth': [3, 5, 7, 10, 15, 20, None],
    'min_samples_split': [2, 5, 10, 15, 20],
    'min_samples_leaf': [1, 2, 4, 6, 8],
    'max_features': ['sqrt', 'log2', None]
}

# Method 2: Using probability distributions (more powerful)
param_dist_continuous = {
    'n_estimators': randint(50, 301),           # Random integers between 50-300
    'max_depth': randint(3, 21),                # Random integers between 3-20 
    'min_samples_split': randint(2, 21),        # Random integers between 2-20
    'min_samples_leaf': randint(1, 9),          # Random integers between 1-8
    'max_features': ['sqrt', 'log2', None]      # Still discrete choices
}

# Create RandomizedSearchCV
random_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_distributions=param_dist_continuous,
    n_iter=100,              # Try 100 random combinations
    cv=5,                    # 5-fold cross-validation
    scoring='accuracy',
    n_jobs=-1,
    verbose=1,
    random_state=42          # For reproducible results
)

# Fit (much faster than GridSearch!)
print("Starting Randomized Search...")
random_search.fit(X_train, y_train)
print("Randomized Search completed!")

# Get results
print(f"\nBest Parameters: {random_search.best_params_}")
print(f"Best CV Score: {random_search.best_score_:.4f}")

### **Comparing GridSearch vs RandomizedSearch**

In [ ]:
import time

# Time comparison
start_time = time.time()
grid_search.fit(X_train, y_train)
grid_time = time.time() - start_time

start_time = time.time()
random_search.fit(X_train, y_train)
random_time = time.time() - start_time

print(f"Grid Search: {grid_time:.2f} seconds, Best Score: {grid_search.best_score_:.4f}")
print(f"Random Search: {random_time:.2f} seconds, Best Score: {random_search.best_score_:.4f}")
print(f"Speedup: {grid_time/random_time:.1f}x faster")

### **When to Use Each Method**

**Use Grid Search when:**
- Small parameter space (< 100 combinations)
- You want to try specific, carefully chosen values
- You have plenty of computational time
- You need to exhaustively explore the space

**Use Randomized Search when:**
- Large parameter space (> 100 combinations)
- You want to explore continuously-valued parameters
- Limited computational time/budget
- Initial exploration of parameter importance

### **Knowledge Check Questions (8.2.4)**

1. **Efficiency Analysis:** You have a parameter space with 10,000 possible combinations. Compare Grid Search vs. Randomized Search with `n_iter=100`. How many combinations does each method try?

2. **Distribution Understanding:** What's the advantage of using `randint(50, 301)` instead of `[50, 100, 150, 200, 250, 300]` for the `n_estimators` parameter?

3. **Strategy Selection:** For each scenario, would you choose Grid Search or Randomized Search?
   - Tuning 2 hyperparameters, each with 3 possible values
   - Tuning 5 hyperparameters, some continuous, with a large search space
   - Fine-tuning around parameters you've already roughly identified

4. **Practical Application:** Your RandomizedSearch with `n_iter=50` finds a CV score of 0.89, while GridSearch on a smaller space finds 0.88. Which result do you trust more and why?


## **8.2.5 Best Practices & Computational Considerations**

Let's cover important practical considerations for effective hyperparameter tuning.

### **1. Computational Cost Management**

**Understanding the Cost:**

In [ ]:
# Cost calculation for GridSearchCV
n_combinations = len(param_grid_values)
n_cv_folds = 5
total_model_fits = n_combinations * n_cv_folds

# Example: 36 combinations × 5 folds = 180 model fits
# Large datasets + complex models = hours/days of computation

**Strategies to Reduce Cost:**

In [ ]:
# Strategy 1: Reduce CV folds for initial exploration
grid_search_quick = GridSearchCV(model, param_grid, cv=3, n_jobs=-1)  # Faster

# Strategy 2: Use smaller parameter grids initially
param_grid_coarse = {
    'n_estimators': [50, 100, 200],           # Fewer values
    'max_depth': [5, 10, None]                # Broader ranges
}

# Strategy 3: Use RandomizedSearch for exploration
random_search = RandomizedSearchCV(model, param_dist, n_iter=50, cv=5)

### **2. Avoiding Overfitting to CV Scores**

**The Problem:**
If you try too many hyperparameter combinations, you might "overfit" to your CV scores.

**Solutions:**

In [ ]:
# 1. Hold out a validation set for final model selection
X_train_tune, X_val, y_train_tune, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42
)

# Use X_train_tune for hyperparameter tuning
grid_search.fit(X_train_tune, y_train_tune)

# Validate best model on X_val before final test
val_score = grid_search.best_estimator_.score(X_val, y_val)

# 2. Use nested cross-validation for unbiased estimates (advanced)

### **3. Smart Parameter Range Selection**

**Start Broad, Then Narrow:**

In [ ]:
# Phase 1: Coarse grid (exploration)
param_grid_coarse = {
    'n_estimators': [10, 100, 1000],
    'max_depth': [1, 10, None],
    'min_samples_split': [2, 50, 200]
}

# Find promising regions, then create fine grid
# Phase 2: Fine grid (exploitation)
param_grid_fine = {
    'n_estimators': [80, 100, 120],        # Around best value from coarse
    'max_depth': [8, 10, 12],              # Around best value from coarse  
    'min_samples_split': [40, 50, 60]      # Around best value from coarse
}

### **4. Multiple Scoring Metrics**

In [ ]:
# Consider multiple metrics
from sklearn.model_selection import cross_validate

scoring = ['accuracy', 'precision', 'recall', 'f1']

# Use cross_validate for comprehensive evaluation
cv_results = cross_validate(model, X_train, y_train, cv=5, scoring=scoring)

for metric in scoring:
    scores = cv_results[f'test_{metric}']
    print(f"{metric}: {scores.mean():.4f} ± {scores.std():.4f}")

### **5. Documentation and Reproducibility**

In [ ]:
# Always set random_state for reproducibility
grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    random_state=42,  # For CV splits
    n_jobs=-1
)

# Log your experiments
import json

experiment_log = {
    'model': 'RandomForestClassifier',
    'param_grid': param_grid,
    'cv_folds': 5,
    'best_params': grid_search.best_params_,
    'best_score': grid_search.best_score_,
    'total_combinations': len(grid_search.cv_results_['params'])
}

with open('experiment_log.json', 'w') as f:
    json.dump(experiment_log, f, indent=2)

### **6. Common Pitfalls to Avoid**

**❌ Don't do this:**

In [ ]:
# Pitfall 1: Using test set for hyperparameter selection
best_params = None
best_score = 0
for params in param_combinations:
    model.set_params(**params)
    model.fit(X_train, y_train)
    score = model.score(X_test, y_test)  # ❌ Using test set!
    if score > best_score:
        best_params = params

# Pitfall 2: Not using stratification for imbalanced data
GridSearchCV(model, param_grid, cv=5)  # ❌ Should use cv=StratifiedKFold()

# Pitfall 3: Ignoring computational constraints
huge_grid = {...}  # ❌ 10,000 combinations without considering time

**✅ Do this instead:**

In [ ]:
# ✅ Correct approach
from sklearn.model_selection import StratifiedKFold

# Use stratified CV for classification
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    estimator=model,
    param_grid=reasonable_grid,  # ✅ Reasonable size
    cv=cv,                       # ✅ Stratified CV
    scoring='f1',                # ✅ Appropriate metric
    n_jobs=-1
)

# ✅ Use only training data
grid_search.fit(X_train, y_train)

# ✅ Final evaluation on test set
final_score = grid_search.best_estimator_.score(X_test, y_test)

### **Knowledge Check Questions (8.2.5)**

1. **Resource Planning:** You have a dataset with 100,000 samples and want to tune 4 hyperparameters with 5 values each using 5-fold CV. Estimate the computational cost and suggest ways to make it more manageable.

2. **Overfitting Recognition:** Your GridSearchCV finds a CV score of 0.95, but the final test score is 0.78. What might be causing this gap, and how would you investigate?

3. **Best Practices:** For each scenario, identify what's wrong and suggest improvements:
   - Using accuracy as the only metric for a highly imbalanced medical diagnosis dataset
   - Running GridSearchCV without setting any random_state parameters
   - Using the test set to compare different hyperparameter combinations

4. **Practical Strategy:** You're starting hyperparameter tuning for a new problem. Outline a step-by-step approach from initial exploration to final tuning.

## **Summary & Transition to Note 8.3**

### **🎯 Key Takeaways from Hyperparameter Tuning**

1. **Hyperparameters control learning** and often determine the difference between good and great models
2. **Systematic search beats manual guessing** - use GridSearchCV or RandomizedSearchCV
3. **Cross-validation is essential** for unbiased hyperparameter selection
4. **Computational efficiency matters** - start broad, then narrow down, consider RandomizedSearch for large spaces
5. **Avoid overfitting to CV scores** by being mindful of search space size and validation strategies

### **🔗 Connection to Your Learning Journey**

- **Days 1-7:** You learned algorithms with default settings
- **Day 8.1:** You learned to evaluate models reliably with cross-validation  
- **Day 8.2:** You now know how to systematically optimize model performance
- **Coming Next:** How to automate the entire workflow and prevent data leakage

### **➡️ Transition to Note 8.3: Pipeline Automation**

You now have two powerful tools:
1. **Cross-validation** for reliable evaluation (Note 8.1)
2. **Hyperparameter tuning** for optimization (Note 8.2)

But there's a critical issue we need to address: **What happens when you need to preprocess your data (scaling, imputation, encoding) AND do hyperparameter tuning?**

In Note 8.3, you'll learn:
- How preprocessing can cause data leakage during CV
- How Scikit-learn Pipelines solve this problem elegantly
- How to combine preprocessing + models + hyperparameter tuning safely
- Best practices for end-to-end ML workflows

**The Problem Preview:** If you scale your data BEFORE doing GridSearchCV, you're accidentally using information from validation folds during preprocessing - that's data leakage! Pipelines fix this.

**🚀 Ready for Note 8.3? Let's learn how to build bulletproof ML workflows!**